## Vercel AI SDK + Neo4j — Environment Setup

This notebook handles **environment setup only** (Node.js packages, credentials, MCP server).

The interactive JavaScript demos live in the **[Observable notebook](https://observablehq.com/import?url=https://raw.githubusercontent.com/neo4j-labs/neo4j-agent-integrations/vercel-agent/vercel-agent/vercel_agent.js)**.


### 1. Install Packages


In [ ]:
import os, subprocess, sys, shutil

node = shutil.which('node')
if not node:
    b = os.path.expanduser('~/node20/bin')
    if os.path.isdir(b):
        os.environ['PATH'] = b + ':' + os.environ.get('PATH', '')
        node = shutil.which('node')
if node:
    v = subprocess.run(['node','--version'],capture_output=True,text=True).stdout.strip()
    print(f'Node.js {v}  ({node})')
else:
    print('ERROR: Node.js not found.')

pkg_dir = os.getcwd()
if not os.path.isdir(os.path.join(pkg_dir,'node_modules','@ai-sdk','mcp')):
    r = subprocess.run(['npm','install'],cwd=pkg_dir,env=os.environ,capture_output=True,text=True)
    print(r.stdout[-400:] if r.stdout else r.stderr[-400:])
else:
    print('npm packages already installed ✓')

r3 = subprocess.run([sys.executable,'-m','pip','install','-q','--ignore-requires-python','neo4j-mcp-server'],capture_output=True,text=True)
print('neo4j-mcp-server:','installed ✓' if r3.returncode==0 else r3.stderr[:200])


### 2. Credentials

The **companies demo database** is public — no changes needed.  
For the **memory agent**, enter credentials in the Observable notebook.


In [ ]:
import os
from getpass import getpass
os.environ.setdefault('NEO4J_URI','neo4j+s://demo.neo4jlabs.com:7687')
os.environ.setdefault('NEO4J_USERNAME','companies')
os.environ.setdefault('NEO4J_PASSWORD','companies')
os.environ.setdefault('AI_PROVIDER','openai')
os.environ.setdefault('AI_MODEL','gpt-4o')
k={'openai':'OPENAI_API_KEY','google':'GOOGLE_GENERATIVE_AI_API_KEY','anthropic':'ANTHROPIC_API_KEY','mistral':'MISTRAL_API_KEY'}
ek=k.get(os.environ['AI_PROVIDER'],'OPENAI_API_KEY')
if not os.environ.get(ek): os.environ[ek]=getpass(f'{ek}: ')
print(f"Provider: {os.environ['AI_PROVIDER']} / {os.environ['AI_MODEL']}")
print(f"Neo4j:    {os.environ['NEO4J_URI']}")


### 3. MCP Server

Start `neo4j-mcp-server` as a background HTTP process. Required only for `1-mcp-agent.mjs`.


In [ ]:
import subprocess, os, time, signal
MCP_PORT = int(os.environ.get('MCP_PORT','8443'))
kill = subprocess.run(['lsof','-ti',f'tcp:{MCP_PORT}'],capture_output=True,text=True)
for pid in kill.stdout.split():
    try: os.kill(int(pid),signal.SIGTERM)
    except ProcessLookupError: pass
os.environ['MCP_PORT']=str(MCP_PORT)
proc = subprocess.Popen(
    ['python','-m','neo4j_mcp',
     '--uri',os.environ['NEO4J_URI'],
     '--username',os.environ['NEO4J_USERNAME'],
     '--password',os.environ['NEO4J_PASSWORD'],
     '--port',str(MCP_PORT),'--transport','http'],
    stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
print(f'neo4j-mcp-server PID {proc.pid} on :{MCP_PORT} ✓')


### 4. MCP Agent (Node.js)


In [ ]:
import subprocess, os
r = subprocess.run(['node','1-mcp-agent.mjs'],cwd=os.getcwd(),env=os.environ,capture_output=True,text=True,timeout=120)
print(r.stdout)
if r.stderr: print('STDERR:',r.stderr[:500])


### 5. Open the Observable Notebook

Direct queries, custom tools, and memory agent all run interactively in the browser:

👉 **[Open Observable Notebook](https://observablehq.com/import?url=https://raw.githubusercontent.com/neo4j-labs/neo4j-agent-integrations/vercel-agent/vercel-agent/vercel_agent.js)**

The notebook prompts for Neo4j URI, username, password, and OpenAI API key.
